In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from dotenv import load_dotenv
import os
import matplotlib.colors as mcolors

In [ ]:
load_dotenv('vars.env')

STAT_PATH = os.getenv("STAT_PATH")

# Changing the reference label will change the reference for all plots and statistics
REF_LABEL = "PAC"

# Plot settings
X_LIM = (20, 180)
Y_LIM = (-75, 75)

In [ ]:
data     = np.load(os.path.join(STAT_PATH, 'bland_altman_density.npy'))
indices  = np.load(os.path.join(STAT_PATH, 'npy_files', 'indices_test_x.npy'))
subj_ids = np.load(os.path.join(STAT_PATH, 'npy_files', 'np_c_finetune.npy'))[indices]

mean_vals = data[0]
diff      = data[1]
test_c    = data[2]

assert len(diff) == len(subj_ids), (
    f"Lengths do not match: diff={len(diff)}, subj_ids={len(subj_ids)}"
)

print(f"Number of measurements:   {len(diff)}")
print(f"Number of patients:  {pd.Series(subj_ids).nunique()}")

### <center><b> Changing bland-altman to repeated measures


In [ ]:
def bland_altman_stats(diff, subject_ids):
    
    df = pd.DataFrame({"subject": subject_ids, "diff": diff})

    subject_means = df.groupby("subject")["diff"].mean()
    bias = subject_means.mean()

    within_var = df.groupby("subject")["diff"].var().mean()
    sd_within  = np.sqrt(within_var)

    loa_upper = bias + 1.96 * sd_within
    loa_lower = bias - 1.96 * sd_within

    n      = len(subject_means)
    t_crit = stats.t.ppf(0.975, df=n - 1)

    se_bias = sd_within / np.sqrt(n)
    se_loa  = np.sqrt(3 * sd_within**2 / n)

    values = {
        "bias":      bias,
        "sd_within": sd_within,
        "loa_upper": loa_upper,
        "loa_lower": loa_lower,
        "n_patients": n,
        "bias_ci":      (bias - t_crit * se_bias,      bias + t_crit * se_bias),
        "loa_upper_ci": (loa_upper - t_crit * se_loa,  loa_upper + t_crit * se_loa),
        "loa_lower_ci": (loa_lower - t_crit * se_loa,  loa_lower + t_crit * se_loa),
    }

    table = pd.DataFrame(
        {
            "Waarde (mL)": [bias, loa_upper, loa_lower],
            "95% CI laag": [values["bias_ci"][0], values["loa_upper_ci"][0], values["loa_lower_ci"][0]],
            "95% CI hoog": [values["bias_ci"][1], values["loa_upper_ci"][1], values["loa_lower_ci"][1]],
        },
        index=["Bias", "Upper LoA", "Lower LoA"],
    ).round(3)

    return values, table

In [ ]:
%matplotlib qt
def bland_altman_plot(mean_vals, diff, colors, values, ref_label="Flotrac",
                      x_lim=(20, 180), y_lim=(-75, 75),color_label="Patient ID" ):
    
    """Bland-Altman scatterplot met bias- en LoA-lijnen."""
    bias      = values["bias"]
    loa_upper = values["loa_upper"]
    loa_lower = values["loa_lower"]

    fig, ax = plt.subplots()
    colors_arr = np.asarray(colors)
    unique_vals = np.unique(colors_arr)
    is_discrete = (len(unique_vals) <= 20 and
                   np.issubdtype(colors_arr.dtype, np.integer))

    if is_discrete:
        import matplotlib.patches as mpatches
        cmap = plt.get_cmap("tab20", len(unique_vals))
        color_map = {v: cmap(i) for i, v in enumerate(unique_vals)}
        point_colors = [color_map[c] for c in colors_arr]
        ax.scatter(mean_vals, diff, c=point_colors, edgecolor="none", s=12)
        patches = [mpatches.Patch(color=color_map[v], label=str(v))
                   for v in unique_vals]
        ax.legend(handles=patches, title=color_label,
                  bbox_to_anchor=(1.01, 1), loc="upper left",
                  borderaxespad=0, fontsize=8, title_fontsize=9)
    else:
        # Continue colorbar
        sc = ax.scatter(mean_vals, diff, c=colors_arr, cmap="viridis",
                        edgecolor="none", s=12)
        cbar = fig.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label("Density", fontsize=10)
        cbar.set_ticks([colors_arr.min(), colors_arr.max()])
        cbar.set_ticklabels(["Low", "High"])

    ax.axhline(bias,      color="gray", linestyle="-", linewidth=0.7)
    ax.axhline(loa_upper, color="gray", linestyle=":", linewidth=1)
    ax.axhline(loa_lower, color="gray", linestyle=":", linewidth=1)

    bbox = {"facecolor": "w", "alpha": 0.7, "linestyle": ""}
    x_text = x_lim[1] - 2
    ax.text(x_text,  loa_upper + 4, f"+1.96 SD: {loa_upper:.3f}", fontsize=10, ha="right", bbox=bbox)
    ax.text(x_text,  bias - 8,      f"bias:\n{bias:.3f}",         fontsize=10, ha="right", bbox=bbox)
    ax.text(x_text,  loa_lower - 8, f"-1.96 SD: {loa_lower:.3f}", fontsize=10, ha="right", bbox=bbox)

    ax.set_xlabel(rf"(SV$_{{APCONet}}$ + SV$_{{{ref_label}}}$)/2 (mL)")
    ax.set_ylabel(rf"SV$_{{APCONet}}$ - SV$_{{{ref_label}}}$ (mL)")
    ax.set_xlim(*x_lim)
    ax.set_ylim(*y_lim)

    plt.tight_layout()
    plt.show()
    return fig, ax

### <center><b>Statistic results

In [ ]:
values, results = bland_altman_stats(diff, subj_ids)

print(f"Number of patients: {values['n_patients']}")
print(f"SD within:        {values['sd_within']:.3f} mL\n")
print(results.to_string())

In [ ]:
bland_altman_plot(mean_vals, diff, test_c, values,
                  ref_label="x", x_lim=X_LIM, y_lim=Y_LIM);

### <center><b>PBAR/AXES Correlation Plot

In [ ]:
data = np.load(os.path.join(STAT_PATH, 'scatter_density.npy'))

flo_trac = data[0]
apconet = data[1]
test_c = data[2]

# Correlaties
rho, pval_s = stats.spearmanr(flo_trac, apconet)
r, pval_p = stats.pearsonr(flo_trac, apconet)

fig, ax = plt.subplots()

sc = ax.scatter(
    x=flo_trac,
    y=apconet,
    c=test_c,
    cmap="viridis",
    edgecolor="none",
    s=12
)

cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Density", fontsize=10)
cbar.set_ticks([test_c.min(), test_c.max()])
cbar.set_ticklabels(["Low", "High"])

ax.axline((0, 0), slope=1, linestyle="--", color="black", alpha=0.5)

ax.set_xlabel(r"SV$_{x}$ (mL)")
ax.set_ylabel(r"SV$_{APCONet}$ (mL)")

ax.set_xlim(0, 200)
ax.set_ylim(0, 200)

# Voeg correlaties toe rechtsonder
ax.text(
    0.97, 0.03,
    rf"Pearson's $r = {r:.2f}$" "\n"
    rf"Spearman's $\rho = {rho:.2f}$",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=10
)

plt.tight_layout()
plt.show()

### <center><b> Replace Axes, 4-PLOT


In [ ]:
indices = np.load(os.path.join(STAT_PATH, 'npy_files', 'indices_test_x.npy'))
svs     = np.load(os.path.join(STAT_PATH, 'SV_predictions.npy'))
test_c  = np.load(os.path.join(STAT_PATH, 'npy_files', 'np_c_finetune.npy'))[indices]

flo_trac = svs[:, 0]
apconet  = svs[:, 1]

pred_diff  = []
true_diff  = []
color_diff = []

for patient in np.unique(test_c):
    pat_true = flo_trac[test_c == patient]
    pat_pred = apconet[test_c == patient]
    for i in range(len(pat_pred) - 1):
        true_diff.append((pat_true[i + 1] - pat_true[i]) / pat_true[i] * 100)
        pred_diff.append((pat_pred[i + 1] - pat_pred[i]) / pat_pred[i] * 100)
        color_diff.append(patient)

ex = 10
excluded     = [(x, y) for x, y in zip(true_diff, pred_diff) if abs(x) <= ex and abs(y) <= ex]
q1           = [(x, y) for x, y in zip(true_diff, pred_diff) if (x < -ex and y >= 0) or (x < 0 and y > ex)]
q2           = [(x, y) for x, y in zip(true_diff, pred_diff) if (x > ex and y >= 0) or (x >= 0 and y > ex)]
q3           = [(x, y) for x, y in zip(true_diff, pred_diff) if (x < -ex and y < 0) or (x < 0 and y < -ex)]
q4           = [(x, y) for x, y in zip(true_diff, pred_diff) if (x > ex and y < 0) or (x >= 0 and y < -ex)]
agreement    = q2 + q3
disagreement = q1 + q4

if len(excluded) + len(agreement) + len(disagreement) != len(true_diff):
    raise ValueError("Data points are not correctly categorized")

ccr           = len(agreement) / (len(agreement) + len(disagreement)) * 100
perc_included = (len(agreement) + len(disagreement)) / len(true_diff) * 100

fig, ax = plt.subplots()

ax.scatter(true_diff, pred_diff, c=color_diff, cmap="Spectral", edgecolor="none", s=12, alpha=0.5)
ax.vlines(0, -50, 50, color="gray", linestyle="--", linewidth=1)
ax.hlines(0, -50, 50, color="gray", linestyle="--", linewidth=1)

exclusion_zone = plt.Rectangle((-ex, -ex), ex * 2, ex * 2,
                                edgecolor=(1, 0, 0, 1), linestyle="--", linewidth=1,
                                fill=True, facecolor=(1, 0, 0, 0.3))
ax.add_patch(exclusion_zone)

ax.set_xlabel(r"ΔSV$_{X}$ (%)")
ax.set_ylabel(r"ΔSV$_{APCONet}$ (%)")
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)

bbox = {'facecolor': 'w', 'alpha': 0.7, 'linestyle': ''}
ax.text(47, -40,
        f"Concordance rate={ccr:.1f}%\n(exclusion zone={ex}%)\n(% included: {perc_included:.1f}%)",
        fontsize=10, ha='right', bbox=bbox)

plt.tight_layout()
plt.show()